# Example 01: Embedded Rebars in Concrete

This notebook demonstrates the creation of a **2D reinforced concrete beam** with explicitly meshed reinforcement bars (steel truss elements) embedded in a concrete continuum (quad elements).

## Key Concepts
- **Conformal mesh**: Uses gmsh's `fragment()` operation to create nodes at rebar-concrete intersections
- **Perfect bond**: Truss elements share nodes with quad elements (no relative slip)
- **Multi-material modeling**: Concrete modeled with 2D nD material (ElasticIsotropic), steel with uniaxial material (Steel01)
- **Physical groups**: Bridge between gmsh regions and OpenSees element/node assignments

## Workflow
1. Build geometry: rectangular concrete domain + three horizontal rebar lines
2. Fragment surface by lines (creates conformal mesh topology)
3. Generate mesh with appropriate element sizes
4. Set up OpenSees model: materials, elements, boundary conditions
5. Export to OpenSeesPy and Tcl for analysis

In [ ]:
from apeGmsh import apeGmsh
from apeGmsh import Algorithm2D
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import gmsh

print("Imports successful. Ready to build model.")

## Parameters

Define geometry, material properties, and mesh sizing.

In [ ]:
# GEOMETRY
L = 4.0      # Beam length [m]
H = 0.5      # Beam height [m]
t = 0.3      # Thickness (out-of-plane) [m]

# Rebar positioning
cover = 0.05  # concrete cover to rebar center [m]
y_rebar = cover  # rebar center height from bottom
diam_bar = 0.02  # rebar diameter [m]
A_bar = np.pi * (diam_bar / 2)**2  # rebar cross-section area [m²]

# Number of bottom rebars
n_rebars = 3
x_rebars = np.linspace(cover, L - cover, n_rebars)

print(f"Beam geometry: L={L} m, H={H} m, thickness={t} m")
print(f"Rebar area: A = {A_bar:.6e} m²")
print(f"Rebar positions (x): {x_rebars}")

# MATERIALS
# Concrete (2D nD material)
E_c = 30e9      # Young's modulus [Pa]
nu_c = 0.2      # Poisson's ratio
rho_c = 2400.0  # density [kg/m³]

# Steel rebars (uniaxial Steel01 material)
E_s = 200e9     # Young's modulus [Pa]
fy = 420e6      # yield stress [Pa]
b_hard = 0.01   # strain hardening ratio

print(f"\nConcrete: E={E_c/1e9:.1f} GPa, nu={nu_c}, rho={rho_c} kg/m³")
print(f"Steel:    E={E_s/1e9:.0f} GPa, fy={fy/1e6:.0f} MPa")

# MESHING
lc_concrete = 0.04  # mesh size in concrete region [m]
lc_rebar = 0.025    # mesh size near rebars [m]

print(f"\nMesh sizes: concrete={lc_concrete} m, rebar={lc_rebar} m")

## Geometry: Concrete Beam with Embedded Rebar Lines

Build the 2D domain and rebar lines using OCC primitives.

In [ ]:
with apeGmsh(model_name="BeamWithRebars", verbose=True) as g:
    print("Building geometry...")

    # Define corner points of concrete domain
    p1 = g.model.geometry.add_point(0.0, 0.0, 0.0, lc_concrete)
    p2 = g.model.geometry.add_point(L, 0.0, 0.0, lc_concrete)
    p3 = g.model.geometry.add_point(L, H, 0.0, lc_concrete)
    p4 = g.model.geometry.add_point(0.0, H, 0.0, lc_concrete)

    # Create rectangle from points
    l1 = g.model.geometry.add_line(p1, p2)  # bottom edge
    l2 = g.model.geometry.add_line(p2, p3)  # right edge
    l3 = g.model.geometry.add_line(p3, p4)  # top edge
    l4 = g.model.geometry.add_line(p4, p1)  # left edge

    # Create curve loop and surface
    loop = g.model.geometry.add_curve_loop([l1, l2, l3, l4])
    concrete_surf = g.model.geometry.add_plane_surface(loop)

    print(f"Concrete surface tag: {concrete_surf}")

    # Create rebar lines inside the concrete domain
    rebar_lines = []
    for i, x_r in enumerate(x_rebars):
        px_left = g.model.geometry.add_point(x_r - diam_bar/2, y_rebar, 0.0, lc_rebar)
        px_right = g.model.geometry.add_point(x_r + diam_bar/2, y_rebar, 0.0, lc_rebar)
        rebar_line = g.model.geometry.add_line(px_left, px_right)
        rebar_lines.append(rebar_line)
        print(f"Rebar {i} line tag: {rebar_line}")

    # Sync OCC changes to gmsh kernel
    g.model.sync()

    # Fragment: Create conformal mesh at rebar-concrete interface
    print(f"\nFragmenting surface (tag {concrete_surf}) with {len(rebar_lines)} rebar lines...")

    # Prepare objects (surfaces) and tools (curves)
    objects = [(2, concrete_surf)]  # 2D surface
    tools = [(1, tag) for tag in rebar_lines]  # 1D curves (rebars)

    # Fragment: splits surface at rebar line intersections
    fragment_result = g.model.boolean.fragment(objects, tools)
    print(f"Fragment completed")

    # After fragment, find the main concrete surface
    surfaces_after = gmsh.model.getEntities(2)
    if surfaces_after:
        concrete_surf_new = surfaces_after[0][1]
        print(f"Concrete surface after fragment: {concrete_surf_new}")
    else:
        raise RuntimeError("No surfaces found after fragment!")

    # Get all curves (boundary edges + rebar lines)
    curves_after = gmsh.model.getEntities(1)
    print(f"Curves after fragment: {len(curves_after)} entities")

    print("\nGeometry complete!")


## Meshing

Generate a 2D mesh with conformal elements at the rebar-concrete interface.

In [ ]:
# Continue with the pyGmsh instance (new context)
with apeGmsh(model_name="BeamWithRebars", verbose=False) as g:
    print("Rebuilding geometry and meshing...")

    # Recreate geometry (same as Cell 6)
    p1 = g.model.geometry.add_point(0.0, 0.0, 0.0, lc_concrete)
    p2 = g.model.geometry.add_point(L, 0.0, 0.0, lc_concrete)
    p3 = g.model.geometry.add_point(L, H, 0.0, lc_concrete)
    p4 = g.model.geometry.add_point(0.0, H, 0.0, lc_concrete)

    l1 = g.model.geometry.add_line(p1, p2)
    l2 = g.model.geometry.add_line(p2, p3)
    l3 = g.model.geometry.add_line(p3, p4)
    l4 = g.model.geometry.add_line(p4, p1)

    loop = g.model.geometry.add_curve_loop([l1, l2, l3, l4])
    concrete_surf = g.model.geometry.add_plane_surface(loop)

    rebar_lines = []
    for x_r in x_rebars:
        px_left = g.model.geometry.add_point(x_r - diam_bar/2, y_rebar, 0.0, lc_rebar)
        px_right = g.model.geometry.add_point(x_r + diam_bar/2, y_rebar, 0.0, lc_rebar)
        rebar_line = g.model.geometry.add_line(px_left, px_right)
        rebar_lines.append(rebar_line)

    g.model.sync()

    # Fragment
    objects = [(2, concrete_surf)]
    tools = [(1, tag) for tag in rebar_lines]
    g.model.boolean.fragment(objects, tools)

    # Find concrete surface and left edge after fragment
    surfaces_after = gmsh.model.getEntities(2)
    concrete_surf = surfaces_after[0][1] if surfaces_after else concrete_surf

    # Meshing
    print("Setting mesh parameters...")
    g.mesh.sizing.set_global_size(lc_concrete)
    g.mesh.generation.set_algorithm(0, Algorithm2D.AUTO, dim=2)

    print("Generating 2D mesh...")
    g.mesh.generation.generate(2)

    print("Optimizing mesh...")
    from apeGmsh import OptimizeMethod
    g.mesh.generation.optimize(OptimizeMethod.NETGEN, niter=2)

    print("\nMesh quality report:")
    quality_df = g.mesh.queries.quality_report()
    print(quality_df.to_string())

    print(f"\nMesh generated successfully!")
    print(f"Total nodes: {len(gmsh.model.mesh.getNodes()[0])}")
    elements = gmsh.model.mesh.getElements(dim=2)
    print(f"Total 2D elements: {len(elements[2]) if elements[2] else 0}")

## OpenSees Model Setup

Configure materials, elements, and boundary conditions.

In [ ]:
# Rebuild geometry, mesh, and set up OpenSees in one context
with apeGmsh(model_name="BeamWithRebars", verbose=False) as g:
    # Rebuild geometry
    p1 = g.model.geometry.add_point(0.0, 0.0, 0.0, lc_concrete)
    p2 = g.model.geometry.add_point(L, 0.0, 0.0, lc_concrete)
    p3 = g.model.geometry.add_point(L, H, 0.0, lc_concrete)
    p4 = g.model.geometry.add_point(0.0, H, 0.0, lc_concrete)

    l1 = g.model.geometry.add_line(p1, p2)
    l2 = g.model.geometry.add_line(p2, p3)
    l3 = g.model.geometry.add_line(p3, p4)
    l4 = g.model.geometry.add_line(p4, p1)

    loop = g.model.geometry.add_curve_loop([l1, l2, l3, l4])
    concrete_surf = g.model.geometry.add_plane_surface(loop)

    rebar_lines = []
    for x_r in x_rebars:
        px_left = g.model.geometry.add_point(x_r - diam_bar/2, y_rebar, 0.0, lc_rebar)
        px_right = g.model.geometry.add_point(x_r + diam_bar/2, y_rebar, 0.0, lc_rebar)
        rebar_line = g.model.geometry.add_line(px_left, px_right)
        rebar_lines.append(rebar_line)

    g.model.sync()

    objects = [(2, concrete_surf)]
    tools = [(1, tag) for tag in rebar_lines]
    g.model.boolean.fragment(objects, tools)

    surfaces_after = gmsh.model.getEntities(2)
    concrete_surf = surfaces_after[0][1] if surfaces_after else concrete_surf

    g.mesh.sizing.set_global_size(lc_concrete)
    g.mesh.generation.set_algorithm(0, Algorithm2D.AUTO, dim=2)
    g.mesh.generation.generate(2)

    # Create physical groups for OpenSees assignment
    print("Creating physical groups...")
    g.physical.add_surface([concrete_surf], name="Concrete")

    # Add rebar curves as physical group
    g.physical.add_curve(rebar_lines, name="Rebar_Bot")

    # Add left edge for fixed BC
    g.physical.add_curve([l4], name="Fixed")

    # Add right edge for potential loading
    g.physical.add_curve([l2], name="LoadEdge")

    print("Physical groups created:")
    print(g.physical.summary().to_string())

    # Configure OpenSees model
    print("\nConfiguring OpenSees model...")
    (g.opensees
       .set_model(ndm=2, ndf=2)
       .add_nd_material("Concrete", "ElasticIsotropic", E=E_c, nu=nu_c, rho=rho_c)
       .add_uni_material("Steel", "Steel01", Fy=fy, E=E_s, b=b_hard)
       .assign_element("Concrete", "quad", material="Concrete", thick=t, eleType="PlaneStress")
       .assign_element("Rebar_Bot", "truss", material="Steel", A=A_bar)
       .fix("Fixed", dofs=[1, 1])
       .build()
    )

    print("\nOpenSees model summary:")
    print(g.opensees.summary())

    # Display node and element tables
    df_nodes = g.opensees.node_table()
    df_elems = g.opensees.element_table()

    print(f"\nNodes: {len(df_nodes)}")
    print(f"Elements: {len(df_elems)}")
    print(f"\nElement types:")
    print(df_elems.groupby('ops_type').size())

    print(f"\nSample nodes (first 5):")
    print(df_nodes.head())

    print(f"\nSample elements (first 5):")
    print(df_elems.head())

## Export Model

Save the OpenSees model in both Tcl and Python formats.

In [ ]:
# Export with the same context
with apeGmsh(model_name="BeamWithRebars", verbose=False) as g:
    # (rebuild geometry, mesh, and configure as above)
    p1 = g.model.geometry.add_point(0.0, 0.0, 0.0, lc_concrete)
    p2 = g.model.geometry.add_point(L, 0.0, 0.0, lc_concrete)
    p3 = g.model.geometry.add_point(L, H, 0.0, lc_concrete)
    p4 = g.model.geometry.add_point(0.0, H, 0.0, lc_concrete)

    l1 = g.model.geometry.add_line(p1, p2)
    l2 = g.model.geometry.add_line(p2, p3)
    l3 = g.model.geometry.add_line(p3, p4)
    l4 = g.model.geometry.add_line(p4, p1)

    loop = g.model.geometry.add_curve_loop([l1, l2, l3, l4])
    concrete_surf = g.model.geometry.add_plane_surface(loop)

    rebar_lines = []
    for x_r in x_rebars:
        px_left = g.model.geometry.add_point(x_r - diam_bar/2, y_rebar, 0.0, lc_rebar)
        px_right = g.model.geometry.add_point(x_r + diam_bar/2, y_rebar, 0.0, lc_rebar)
        rebar_line = g.model.geometry.add_line(px_left, px_right)
        rebar_lines.append(rebar_line)

    g.model.sync()

    objects = [(2, concrete_surf)]
    tools = [(1, tag) for tag in rebar_lines]
    g.model.boolean.fragment(objects, tools)

    surfaces_after = gmsh.model.getEntities(2)
    concrete_surf = surfaces_after[0][1] if surfaces_after else concrete_surf

    g.mesh.sizing.set_global_size(lc_concrete)
    g.mesh.generation.set_algorithm(0, Algorithm2D.AUTO, dim=2)
    g.mesh.generation.generate(2)

    g.physical.add_surface([concrete_surf], name="Concrete")
    g.physical.add_curve(rebar_lines, name="Rebar_Bot")
    g.physical.add_curve([l4], name="Fixed")
    g.physical.add_curve([l2], name="LoadEdge")

    (g.opensees
       .set_model(ndm=2, ndf=2)
       .add_nd_material("Concrete", "ElasticIsotropic", E=E_c, nu=nu_c, rho=rho_c)
       .add_uni_material("Steel", "Steel01", Fy=fy, E=E_s, b=b_hard)
       .assign_element("Concrete", "quad", material="Concrete", thick=t, eleType="PlaneStress")
       .assign_element("Rebar_Bot", "truss", material="Steel", A=A_bar)
       .fix("Fixed", dofs=[1, 1])
       .build()
    )

    # Export
    print("Exporting OpenSees model...")
    g.opensees.export_tcl("01_beam_rebars.tcl")
    g.opensees.export_py("01_beam_rebars.py")

    print("\nExport complete:")
    print("  - 01_beam_rebars.tcl  (OpenSees Tcl script)")
    print("  - 01_beam_rebars.py   (OpenSeesPy Python script)")

    # Show first 50 lines of generated Python script
    print("\n--- First 50 lines of 01_beam_rebars.py ---\n")
    try:
        with open("01_beam_rebars.py", "r") as f:
            lines = f.readlines()[:50]
            print("".join(lines))
    except FileNotFoundError:
        print("(File not yet written to disk)")

## OpenSeesPy Analysis

Run a static gravity analysis. Note: openseespy must be installed.

In [ ]:
# Attempt to import openseespy; if not available, show the concept
try:
    import openseespy.opensees as ops
    HAS_OPENSEESPY = True
except ImportError:
    print("openseespy not installed. Showing analysis concept.")
    HAS_OPENSEESPY = False

if HAS_OPENSEESPY:
    print("Running OpenSeesPy analysis...")

    # Read and execute the exported model
    exec(open("01_beam_rebars.py").read())

    # Apply self-weight (body force already included in element definition)
    # For demonstration, apply a concentrated load at mid-span
    ops.timeSeries('Linear', 1)
    ops.pattern('Plain', 1, 1)

    # Find mid-span nodes (approximately at x = L/2)
    nodes = ops.getNodeTags()
    mid_nodes = []
    for nid in nodes:
        crd = ops.nodeCoord(nid)
        if len(crd) >= 1 and abs(crd[0] - L/2) < 0.1:
            mid_nodes.append(nid)

    if mid_nodes:
        # Apply downward load to first mid-span node found
        ops.load(mid_nodes[0], 0.0, -10000.0)  # 10 kN downward

    # Solver configuration
    ops.constraints('Transformation')
    ops.numberer('RCM')
    ops.system('BandGeneral')
    ops.test('NormDispIncr', 1e-6, 100)
    ops.algorithm('Newton')
    ops.integrator('LoadControl', 0.1)
    ops.analysis('Static')

    # Run analysis
    print("Analyzing structure (10 load steps)...")
    ok = ops.analyze(10)

    if ok == 0:
        print("Analysis completed successfully!")
        if len(nodes) > 0:
            test_node = nodes[-1]
            disp = ops.nodeDisp(test_node)
            print(f"\nNode {test_node} displacement: {disp}")
    else:
        print("Analysis encountered convergence issues.")
else:
    print("Analysis concept (openseespy not available):")
    print("ops.wipe()")
    print("ops.model('basic', '-ndm', 2, '-ndf', 2)")
    print("# Apply gravity and loadcase...")
    print("ops.analyze(10)")

## Post-Processing and Visualization

Plot the deformed shape and element stresses.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Undeformed mesh and rebar locations
ax1.set_title("Undeformed Mesh with Rebars")
ax1.set_aspect('equal')

# Draw concrete domain
rect = mpatches.Rectangle((0, 0), L, H, linewidth=2, edgecolor='black',
                          facecolor='lightgray', alpha=0.3, label='Concrete domain')
ax1.add_patch(rect)

# Draw rebars
for i, x_r in enumerate(x_rebars):
    ax1.plot([x_r - diam_bar/2, x_r + diam_bar/2], [y_rebar, y_rebar],
            'r-', linewidth=3, label='Rebar' if i == 0 else '')
    ax1.plot(x_r, y_rebar, 'ro', markersize=8)

# Fixed support
ax1.plot([0, 0], [0, H], 'g-', linewidth=2, label='Fixed edge')
ax1.plot([0, 0.1], [-0.05, -0.05], 'g-', linewidth=1)
ax1.plot([0.05, 0.05], [-0.03, -0.07], 'g-', linewidth=1)

ax1.set_xlim(-0.3, L + 0.3)
ax1.set_ylim(-0.15, H + 0.15)
ax1.set_xlabel('X [m]')
ax1.set_ylabel('Y [m]')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper right')

# Plot 2: Material properties summary
ax2.axis('off')
diam_str = f"{diam_bar*1000:.1f}"
area_str = f"{A_bar*1e4:.2f}"
cover_str = f"{cover*1000:.0f}"
ec_str = f"{E_c/1e9:.1f}"
es_str = f"{E_s/1e9:.0f}"
fy_str = f"{fy/1e6:.0f}"

summary_text = "BEAM WITH EMBEDDED REBARS - Summary\n\n"
summary_text += f"GEOMETRY:\n"
summary_text += f"  Length: {L} m\n"
summary_text += f"  Height: {H} m\n"
summary_text += f"  Thickness: {t} m\n\n"
summary_text += f"REBARS (Bottom):\n"
summary_text += f"  Count: {n_rebars}\n"
summary_text += f"  Diameter: {diam_str} mm\n"
summary_text += f"  Area each: {area_str} cm²\n"
summary_text += f"  Cover: {cover_str} mm\n\n"
summary_text += f"CONCRETE:\n"
summary_text += f"  E: {ec_str} GPa\n"
summary_text += f"  nu: {nu_c}\n"
summary_text += f"  rho: {rho_c} kg/m³\n\n"
summary_text += f"STEEL REBARS:\n"
summary_text += f"  E: {es_str} GPa\n"
summary_text += f"  fy: {fy_str} MPa\n"
summary_text += f"  Hardening: {b_hard}\n\n"
summary_text += f"MESH:\n"
summary_text += f"  Concrete size: {lc_concrete} m\n"
summary_text += f"  Rebar size: {lc_rebar} m"

ax2.text(0.1, 0.5, summary_text, fontsize=10, family='monospace',
        verticalalignment='center', 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('01_beam_rebars_summary.png', dpi=100, bbox_inches='tight')
plt.show()

print("Summary plot saved: 01_beam_rebars_summary.png")

## Key Takeaways

### Fragment Operation
- **Purpose**: Creates conformal mesh at interface between geometric objects
- **Input**: Objects (concrete surface) and Tools (rebar lines)
- **Output**: Modified surface entities with new tags; nodes at line-surface intersections
- **Result**: Truss and quad elements share nodes for perfect bond

### Element Types
**Concrete**: `quad` (4-node bilinear quadrilateral)
- Uses nD material (ElasticIsotropic in this case)
- Thickness specified for plane stress/strain analysis
- Gmsh element type 3 (4-node quad)

**Rebars**: `truss` (2-node truss bar)
- Uses uniaxial material (Steel01)
- Cross-section area as parameter
- Gmsh element type 1 (2-node line)

### Physical Groups as Bridge
Physical groups assign gmsh geometric regions to OpenSees element templates:
- `Concrete` surface → quad elements with concrete material
- `Rebar_Bot` curves → truss elements with steel material
- `Fixed` curve → boundary condition (nodes constrained)
- `LoadEdge` curve → load application nodes

### Multi-Physics Integration
- **Gmsh** handles mesh generation, topology, region identification
- **pyGmsh** provides clean Python wrapper and entity registry
- **OpenSees** receives node coordinates and element connectivity via export
- Physical groups enable automatic, scalable assignments

### Next Steps
- Implement distributed loading along beam edges
- Add material nonlinearity (ConcreteD, Hardening Steel)
- Run nonlinear analysis (fiber sections, plasticity)
- Export to visualization tools (ParaView, VisIt)